In this note book we procede to axiomatisation.

In [9]:
import os
import spacy
import pandas as pd
from typing import Set, List
from olaf import Pipeline
from olaf.commons.logging_config import logger
from olaf.data_container import CandidateTerm, Relation, Concept
from olaf.data_container.knowledge_representation_schema import KnowledgeRepresentation
from olaf.pipeline.pipeline_component.term_extraction import (
    POSTermExtraction,
    TFIDFTermExtraction,
    ManualCandidateTermExtraction
    )

from olaf.pipeline.pipeline_component.concept_relation_extraction import (
    CTsToConceptExtraction, CTsToRelationExtraction,
    SynonymRelationExtraction, SynonymConceptExtraction,
    AgglomerativeClusteringRelationExtraction, AgglomerativeClusteringConceptExtraction
)
from olaf.pipeline.pipeline_component.axiom_extraction import OWLAxiomExtraction


from olaf.commons.kr_to_rdf_tools import (
    kr_concepts_to_owl_classes, kr_relations_to_owl_obj_props, 
    kr_metarelations_to_owl, kr_relations_to_anonymous_some_parent, concept_lrs_to_owl_individuals
)

from olaf.commons.spacy_processing_tools import is_not_punct, is_not_stopword, select_on_pos

from olaf.pipeline.pipeline_component.candidate_term_enrichment import SemanticBasedEnrichment

from olaf.repository.corpus_loader.text_corpus_loader import TextCorpusLoader

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from matplotlib_venn import venn2, venn3

In [11]:
import torch, gc
def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

In [12]:
nlp = spacy.load("en_core_web_lg")

# Select Corpus

In [13]:
corpus_path = "GC10-DET_doc.txt"
corpus_loader = TextCorpusLoader(corpus_path)

In [14]:
concepts = [
    "defect type",
    "steel strip surface",
    "punching",
    "mechanical failure",
    "welding line",
    "coil",
    "weld line",
    "crescent gap",
    "cutting",
    "water spot",
    "drying",
    "oil spot",
    "mechanical lubricant",
    "silk spot",
    "plaque",
    "strip surface",
    "roller",
    "pressure",
    "inclusion",
    "metal surface",
    "spots",
    "fish scale shape",
    "block irregular distribution",
    "rolled pit",
    "bulges",
    "pits",
    "steel plate",
    "work roll",
    "tension roll",
    "damage",
    "crease",
    "fold",
    "uncoiling process",
    "waist folding",
    "deformation",
    "low-carbon"
]

ct_concept_label = { concept : {concept} for concept in concepts}


llm_output = [
    ["water spot", "is produced by", "drying"],
    ["water spot", "is produced by", "production"],
    ["oil spot", "is caused by", "contamination"],
    ["contamination", "is caused by", "mechanical lubricant"],
    ["oil spot", "has appearance", "product"],
    ["crescent gap", "is caused by", "cutting"],
    ["weld line", "is part of", "strip"],
    ["inclusion", "has appearance", "small spots"],
    ["inclusion", "has appearance", "fish scale shape"],
    ["inclusion", "has appearance", "strip shape"],
    ["inclusion", "has appearance", "block irregular distribution"],
    ["inclusion", "is part of", "upper surface"],
    ["inclusion", "is part of", "lower surface"],
    ["inclusion", "is accompanied by", "rough pockmarked surfaces"],
    ["crease", "has appearance", "vertical transverse fold"],
    ["crease", "has abnormal", "spacing"],
    ["crease", "is caused by", "local yield"],
    ["crease", "is part of", "strip"],
    ["silk spot", "has appearance", "plaque"],
    ["silk spot", "is caused by", "uneven temperature"],
    ["silk spot", "is caused by", "uneven pressure"],
    ["waist folding", "has appearance", "obvious folds"],
    ["waist folding", "has appearance", "wrinkles"],
    ["waist folding", "is caused by", "local deformation"],
    ["waist folding", "is caused by", "low-carbon"],
    ["punching", "is produced by", "production line"],
    ["punching", "is produced by", "strip"],
    ["punching", "is caused by", "mechanical failure"],
    ["punctate", "is part of", "rolled pit"],
    ["rolled pit", "has appearance", "bulges"],
    ["rolled pit", "has appearance", "pits"],
    ["rolled pit", "is caused by", "work roll"],
    ["rolled pit", "is caused by", "tension roll"],
    ["rolled pit", "is part of", "steel plate"]
]

found_relations = {
   Relation(
       relation[1], 
       Concept(relation[0]), 
       Concept(relation[2])
       ) for relation in llm_output
}

seed_kr = KnowledgeRepresentation(
    relations=found_relations
)

In [15]:

axiom_generators = {    
        kr_concepts_to_owl_classes,
        kr_relations_to_owl_obj_props,
        kr_metarelations_to_owl,
        kr_relations_to_anonymous_some_parent,
        concept_lrs_to_owl_individuals
    }

owl_axiom_extraction = OWLAxiomExtraction(
    owl_axiom_generators=axiom_generators,
    base_uri="https://github.com/wikit-ai/olaf-llm-eswc2024/o/example#"
)

In [16]:


pipeline = Pipeline(
    seed_kr=seed_kr,
    spacy_model=nlp,
    pipeline_components=[
        ManualCandidateTermExtraction(
            ct_label_strings_map=ct_concept_label
        ),
        AgglomerativeClusteringConceptExtraction(
            distance_threshold=.4
        ),
        owl_axiom_extraction
    ],
    corpus_loader=corpus_loader
)

pipeline.run()

[2024-07-08 17:00:22,686] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-08 17:00:22,687] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]


In [17]:
kr_rdf_graph_path = os.path.join("complete_pipeline_kr_rdf_graph.ttl")
pipeline.kr.rdf_graph.serialize(kr_rdf_graph_path, format="ttl")

<Graph identifier=Nb7705b3653c2477cbff44b6778753423 (<class 'rdflib.graph.Graph'>)>